In [ ]:
!nvidia-smi
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece openai

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# MiniMax M2.5 Colab Notebook

Bu notebook su adimlari gosterir:
1. Ornek veri yukleme
2. MiniMax-M2.5 ile metin uretimi (pipeline ve direct model)
3. Hugging Face Router uzerinden API kullanimi
4. Opsiyonel LoRA egitimi ve cikti kaydetme

In [ ]:
import json
import os
import torch
from datasets import load_dataset

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Sat Mar 21 15:35:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1) Veri yukleme

Bu ornekte once kucuk bir ornek egitim verisi olusturup JSONL dosyasi olarak kaydediyoruz, sonra datasets ile yukluyoruz.

In [12]:
import json
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Ornek egitim verisi
sample_rows = [
    {
        'instruction': 'Translate to English',
        'input': 'Merhaba dunya',
        'output': 'Hello world'
    },
    {
        'instruction': 'Summarize the text',
        'input': 'LLaMA is a family of language models by Meta.',
        'output': 'LLaMA is a Meta language model family.'
    },
    {
        'instruction': 'Answer the question',
        'input': 'What is 2 + 2?',
        'output': '4'
    }
]

os.makedirs('/content/data', exist_ok=True)
train_path = '/content/data/train.jsonl'

with open(train_path, 'w', encoding='utf-8') as f:
    for row in sample_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

raw_dataset = load_dataset('json', data_files=train_path, split='train')
raw_dataset

CUDA available: True
GPU: Tesla T4


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 3
})

## 2) MiniMax-M2.5 ile metin uretimi

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline('text-generation', model='MiniMaxAI/MiniMax-M2.5', trust_remote_code=True)
messages = [
    {'role': 'user', 'content': 'Who are you?'},
]
pipe(messages, max_new_tokens=40)

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained('MiniMaxAI/MiniMax-M2.5', trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained('MiniMaxAI/MiniMax-M2.5', trust_remote_code=True)

In [ ]:
messages = [
    {'role': 'user', 'content': 'Who are you?'},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:]))

In [ ]:
from openai import OpenAI

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    raise ValueError('HF_TOKEN ortam degiskenini ayarlayin.')

client = OpenAI(
    base_url='https://router.huggingface.co/v1',
    api_key=hf_token,
)

completion = client.chat.completions.create(
    model='MiniMaxAI/MiniMax-M2.5',
    messages=[
        {
            'role': 'user',
            'content': 'What is the capital of France?'
        }
    ],
)

print(completion.choices[0].message)

## 3) Opsiyonel: LoRA egitimi ve cikti kaydetme

Bu bolum MiniMax modeli ile LoRA egitimi denemesi icin ornek akistir.
Not: Colab T4 gibi ortamlarda bellek yetersizligi olabilir.

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import zipfile

def format_example(x):
    return {
        'text': f"### Instruction:\n{x['instruction']}\n\n### Input:\n{x['input']}\n\n### Response:\n{x['output']}"
    }

dataset = raw_dataset.map(format_example)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

training_args = SFTConfig(
    output_dir='/content/minimax-lora-demo',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy='no',
    max_seq_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
    dataset_text_field='text'
)

train_result = trainer.train()

save_dir = '/content/output/minimax_lora_adapter'
os.makedirs(save_dir, exist_ok=True)
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

metrics_path = '/content/output/train_metrics.json'
os.makedirs('/content/output', exist_ok=True)
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

zip_path = '/content/output/minimax_lora_adapter.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(save_dir):
        for file_name in files:
            full_path = os.path.join(root, file_name)
            arcname = os.path.relpath(full_path, save_dir)
            zf.write(full_path, arcname=arcname)

print('Kaydedilen klasor:', save_dir)
print('Metrik dosyasi:', metrics_path)
print('Zip dosyasi:', zip_path)